# Receptor block ablations — how much does the ESM protein embedding really add?

Same data, folds and features as the **full_full baseline** (`orbind.lorax`): the
LORAX / Hladis M2OR pool, features = **putative ESM-1b (1280-d) ‖ ChemBERTa-77M
(384-d)**. One repeat per regime (`transductive` = LORAX `rand_split_1`, EC50-only
test; `inductive_molecule` = cold-molecule seed 42). Molecule block is fixed; only
the **protein block** changes. Head = plain `train_boost` (single XGBoost), all six
metrics. Runner is incremental + resumable.

**The ladder decomposes the protein signal:**

| variant | protein block | question it answers |
|---|---|---|
| `molecule_only` | *(dropped)* | floor — how much does the protein side add at all? |
| `onehot` | per-receptor identity (`eye`) | pure identity lookup, zero biology |
| `onehot_fam` | identity ⊕ OR-family one-hot | does coarse family add over identity? |
| `onehot_kmeans17` | identity ⊕ ESM cluster-centroid (17) | is ESM's lift **over identity** just coarse? |
| `mean_onehot` | identity ⊕ full ESM-1b | matched ceiling for the two rows above |
| `pca8` | ESM projected to 8 dims | is the 1280-d embedding really ~8-d of usable signal? |
| `mean` | full ESM-1b (identity implicit) | the honest embedding |
| `mlp64` | 64-d penultimate of an MLP trained on `[mol‖ESM]` (head dropped), then boost | can a *supervised* neural bottleneck extract more than raw-embedding+tree? |

Reads: `onehot` is bare identity. `onehot_fam` / `onehot_kmeans17` add family / coarse-ESM
*on top of* identity; if either ≈ `mean_onehot` (identity ⊕ full ESM), then ESM's usable
content is coarse (family-level). If `mean ≈ pca8`, the embedding is low-rank. The gap
`mean − molecule_only` is the protein block's total contribution. (Standalone `kmeans17`
— ESM coarsened with identity *removed* — is left out of the default list; it sits below
`onehot` precisely because it drops identity, so it isn't comparable to it.)


In [ ]:
# --- setup & config -----------------------------------------------------
import pathlib, sys, csv, re, io, zipfile
import numpy as np, pandas as pd

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    if ROOT == ROOT.parent:
        raise RuntimeError("run this notebook from inside the orbind repo")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
DATA = ROOT / "data"

from orbind import lorax as L
from orbind.baselines import train_boost
from orbind.dataset import metrics

# regime -> (build-regime, fold, cold-seed). One repeat each.
REGIMES = {
    "transductive":       ("transductive",       1, None),
    "inductive_molecule": ("inductive_molecule", 1, 42),
}
# trim/extend freely; runner skips finished (variant, regime) rows.
VARIANTS   = ["mean", "onehot", "molecule_only", "onehot_fam",
              "onehot_kmeans17", "mean_onehot", "pca8", "mlp64"]
BOOST_SEED = 42
# mlp{K}: train an MLP on the FULL [molecule || ESM] pair (…->K->2) for many
# epochs, discard the K->2 head, boost on the K-d penultimate representation.
MLP_EPOCHS  = 300
MLP_HIDDEN  = 256
MLP_LR      = 1e-3
MLP_DROPOUT = 0.3
METRIC_KEYS = ["AUROC", "AUPRC", "MCC", "F1", "precision", "recall"]
COLS = ["variant", "regime", "repeat", *METRIC_KEYS, "n_train", "n_test", "test_pos"]

OUT = ROOT / "results" / "full_full" / "tables" / "receptor_block_ablations.csv"
OUT.parent.mkdir(parents=True, exist_ok=True)
print("OUT:", OUT)


In [ ]:
# --- OR-family label for every receptor (full coverage via blast.csv) ----
# blast.csv (in M2OR.zip) gives a gene name for ~99% of the pool's receptors.
# Human + other-mammal receptors map to a human OR ortholog -> clean OR-family
# (OR1..OR14, OR51/52/56, ...). Mouse (Olfr*) and the handful without a gene get
# a single shared 'OTHER' bucket, by request.
def load_family_fn():
    z = zipfile.ZipFile(DATA / "raw" / "M2OR.zip")
    bl = pd.read_csv(io.BytesIO(z.read("blast.csv")), sep=";")
    seq2gene = dict(zip(bl["mutated_sequence"], bl["blast_gene_name"]))
    def fam(seq):
        g = seq2gene.get(seq)
        if isinstance(g, str) and "OLFR" not in g.upper():
            m = re.search(r"OR(\d+)", g.upper())
            if m:
                return "OR" + m.group(1)
        return "OTHER"          # mouse Olfr + no-gene
    return fam

family_of = load_family_fn()


In [ ]:
# --- assemble helpers ----------------------------------------------------
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

def _xy(split, Xm, Xp):
    # Xp=None -> molecule-only (drop the protein block)
    idx = np.concatenate([split["pos"], split["neg"]], axis=0)
    y = np.concatenate([np.ones(len(split["pos"])), np.zeros(len(split["neg"]))])
    Xm_rows = Xm[idx[:, 0]]
    X = Xm_rows if Xp is None else np.concatenate([Xm_rows, Xp[idx[:, 1]]], axis=1)
    return X.astype(np.float32), y.astype(np.float32)

def prot_sequences(fold):
    # receptor sequence per Xp row index (same node universe L.build uses)
    dfs = L._load_fold(fold, esm, chem)
    _, _, _, pi = L._node_universe(dfs, esm, chem)
    seqs = [None] * len(pi)
    for seq, i in pi.items():
        seqs[i] = seq
    return seqs

def mlp_features(Xtr, ytr, Xte, penult=64, hidden=MLP_HIDDEN, epochs=MLP_EPOCHS,
                 lr=MLP_LR, dropout=MLP_DROPOUT, seed=BOOST_SEED):
    # train MLP (D -> hidden -> penult -> 2) on binding, then return the penult-d
    # activations for train/test (the 2-logit head is discarded).
    import torch, torch.nn as nn
    torch.manual_seed(seed); np.random.seed(seed)
    mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-6
    Xtr_t = torch.tensor((Xtr - mu) / sd); Xte_t = torch.tensor((Xte - mu) / sd)
    ytr_t = torch.tensor(ytr).long()
    D = Xtr.shape[1]
    backbone = nn.Sequential(nn.Linear(D, hidden), nn.ReLU(), nn.BatchNorm1d(hidden),
                             nn.Dropout(dropout), nn.Linear(hidden, penult), nn.ReLU())
    model = nn.Sequential(backbone, nn.Linear(penult, 2))
    n0, n1 = float((ytr == 0).sum()), float((ytr == 1).sum())
    w = torch.tensor([(n0 + n1) / (2 * max(n0, 1)), (n0 + n1) / (2 * max(n1, 1))], dtype=torch.float)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    lossf = nn.CrossEntropyLoss(weight=w)
    n, batch = len(ytr), 256
    for _ in range(epochs):
        model.train(); perm = torch.randperm(n)
        for i in range(0, n, batch):
            b = perm[i:i + batch]
            if len(b) < 2: continue           # BatchNorm needs >=2 samples
            opt.zero_grad(); lossf(model(Xtr_t[b]), ytr_t[b]).backward(); opt.step()
    model.eval()
    with torch.no_grad():
        return backbone(Xtr_t).numpy().astype(np.float32), backbone(Xte_t).numpy().astype(np.float32)

def protein_block(variant, Xp_esm, prot_seqs):
    # return the [n_receptor, d] protein block for a variant (or None to drop it)
    n = Xp_esm.shape[0]
    if variant == "mean":          return Xp_esm
    if variant == "onehot":        return np.eye(n, dtype=np.float32)
    if variant == "molecule_only": return None
    if variant in ("onehot_fam", "fam_only"):
        fams = [family_of(s) for s in prot_seqs]
        cats = sorted(set(fams)); fi = {f: i for i, f in enumerate(cats)}
        fam_oh = np.eye(len(cats), dtype=np.float32)[[fi[f] for f in fams]]
        return fam_oh if variant == "fam_only" else np.concatenate([np.eye(n, dtype=np.float32), fam_oh], axis=1)
    if variant == "mean_onehot":                     # identity + full ESM (matched ceiling)
        return np.concatenate([np.eye(n, dtype=np.float32), Xp_esm], axis=1)
    if variant.startswith("pca"):
        k = int(variant[3:])
        return PCA(n_components=k, random_state=BOOST_SEED).fit_transform(Xp_esm).astype(np.float32)
    if variant.startswith("onehot_kmeans"):          # identity + coarse (cluster-centroid) ESM
        c = int(variant[len("onehot_kmeans"):])
        km = KMeans(n_clusters=c, n_init=10, random_state=BOOST_SEED).fit(Xp_esm)
        coarse = km.cluster_centers_[km.labels_].astype(np.float32)
        return np.concatenate([np.eye(n, dtype=np.float32), coarse], axis=1)
    if variant.startswith("kmeans"):                 # coarse ESM ONLY (identity destroyed)
        c = int(variant[6:])
        km = KMeans(n_clusters=c, n_init=10, random_state=BOOST_SEED).fit(Xp_esm)
        return km.cluster_centers_[km.labels_].astype(np.float32)
    raise ValueError(variant)


In [ ]:
# --- load features once --------------------------------------------------
esm, chem = L.load_embeddings()      # ESM-1b (seq->1280)  |  ChemBERTa (SMILES->384)
print(f"protein={len(esm)}x{next(iter(esm.values())).shape[0]}  "
      f"molecule={len(chem)}x{next(iter(chem.values())).shape[0]}")

# family-coverage sanity print on fold 1
_seqs = prot_sequences(1)
_fam = pd.Series([family_of(s) for s in _seqs])
print(f"receptors={len(_seqs)}  OR-families={_fam.nunique()-(1 if 'OTHER' in _fam.values else 0)}  "
      f"OTHER bucket={(_fam=='OTHER').sum()} ({100*(_fam=='OTHER').mean():.0f}%)")


In [ ]:
# --- run: variant x regime, incremental + resumable ---------------------
def load_done(path):
    if not path.exists(): return set()
    return {(r.variant, r.regime) for r in pd.read_csv(path).itertuples()}

def append_row(path, row):
    new = not path.exists()
    with open(path, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=COLS)
        if new: w.writeheader()
        w.writerow(row)

done = load_done(OUT)
print(f"resuming: {len(done)} rows already on disk\n")

for regime, (build_regime, fold, seed) in REGIMES.items():
    Xm, Xp, splits = L.build(build_regime, fold, esm, chem, seed=(seed or BOOST_SEED))
    Xm_np, Xp_esm = Xm.numpy(), Xp.numpy()
    prot_seqs = prot_sequences(fold)
    for variant in VARIANTS:
        if (variant, regime) in done:
            continue
        if variant.startswith("mlp"):
            # supervised neural bottleneck on the FULL [mol||ESM] pair, then boost on it
            penult = int(variant[3:]) if variant[3:].isdigit() else 64
            Xtr, ytr = _xy(splits["train"], Xm_np, Xp_esm)
            Xte, yte = _xy(splits["test"],  Xm_np, Xp_esm)
            Ztr, Zte = mlp_features(Xtr, ytr, Xte, penult=penult)
            score = train_boost(Ztr, ytr, Zte, seed=BOOST_SEED)
        else:
            block = protein_block(variant, Xp_esm, prot_seqs)
            Xtr, ytr = _xy(splits["train"], Xm_np, block)
            Xte, yte = _xy(splits["test"],  Xm_np, block)
            score = train_boost(Xtr, ytr, Xte, seed=BOOST_SEED)
        m = metrics(yte, score)
        row = {"variant": variant, "regime": regime,
               "repeat": fold if seed is None else seed,
               **{k: round(float(m[k]), 6) for k in METRIC_KEYS},
               "n_train": int(len(ytr)), "n_test": int(len(yte)), "test_pos": int(yte.sum())}
        append_row(OUT, row); done.add((variant, regime))
        print(f"  [{variant:<13} | {regime:<18}] " +
              " ".join(f"{k}={m[k]:.3f}" for k in METRIC_KEYS), flush=True)
print("\ndone.")


In [ ]:
# --- summary + how-weak decomposition -----------------------------------
res = pd.read_csv(OUT)
present = [v for v in VARIANTS if v in set(res["variant"])]
for metric in METRIC_KEYS:
    piv = res.pivot_table(index="variant", columns="regime", values=metric).reindex(present)[list(REGIMES)]
    print(f"\n### {metric}")
    print(piv.round(3).to_string())

# explicit decomposition on AUROC/AUPRC/MCC: how much each block recovers between
# the molecule-only floor and full ESM.
print("\n" + "=" * 60)
for metric in ["AUROC", "AUPRC", "MCC"]:
    piv = res.pivot_table(index="variant", columns="regime", values=metric)
    if not {"molecule_only", "mean"} <= set(piv.index): continue
    lo, hi = piv.loc["molecule_only"], piv.loc["mean"]
    span = (hi - lo).replace(0, np.nan)
    frac = piv.subtract(lo).div(span)   # 0 = molecule floor, 1 = full ESM
    frac = frac.reindex(present)[list(REGIMES)]
    print(f"\n### {metric}: fraction of the (mean - molecule_only) gap recovered")
    print(f"    protein contribution mean-molecule_only: " +
          " ".join(f"{r}={float(hi[r]-lo[r]):+.3f}" for r in REGIMES))
    print(frac.round(2).to_string())


## Notes

* **Honest ESM here = ESM-1b** (`esm1b_650m_mean.npz`), the vector the full_full /
  ensemble pipeline actually uses.
* **Family label** comes from `blast.csv` in `M2OR.zip` (`blast_gene_name` → `OR\d+`),
  full coverage: human + other mammals → one of ~18 OR families; **mouse (Olfr*) and
  the ~8 gene-less receptors share a single `OTHER` bucket** (mouse is ~13% of unique
  receptors and the EC50 test is human-dominated, so this is a safe simplification).
* **Resumable**: kill anytime; re-running skips finished `(variant, regime)` rows.
  Delete `results/full_full/tables/receptor_block_ablations.csv` to start clean.
* PCA/KMeans are fit unsupervised on the receptor vectors (no labels), so they leak
  nothing. Add points (`pca2`, `pca32`, `kmeans50`, `fam_only`) by editing `VARIANTS`.
* Fixed XGBoost hyperparameters (`train_boost`) to isolate the representation axis.
